# Auto Dealer Customer Support: Case Aggregation & SLA Pipeline

## Project Selection Rationale

This code sample demonstrates a **multi-source case aggregation and SLA compliance pipeline** from a previous data engineering role supporting dealership customer support operations. 

**Why this specific piece of code:**
- It represents a common real-world challenge: consolidating fragmented customer support data from heterogeneous systems (CRM, parts ordering, service ticketing)
- Demonstrates foundational Data Engineering patterns: ETL logic, data quality validation, schema normalization, and metrics calculation
- Directly addresses auto dealer pain points: SLA tracking, case prioritization, and escalation management
- Showcases practical techniques for handling data quality issues common in automotive support systems

## Context

In auto dealer customer support, cases originate from multiple sources:
- **Service Cases**: Generated by service departments when customers call about repairs or maintenance
- **Parts Ordering Cases**: Created when customers or technicians need to order specific parts (VIN-based lookups, compatibility checks)
- **Warranty Cases**: Warranty claims and coverage disputes
- **Card Key Cases**: Issues with key fobs, card access, or key replacement requests

Each system stores data in different schemas and formats. The challenge was creating a unified view for the support team to track all cases, monitor response times, and identify escalations.

## Problem Being Solved

1. **Data Fragmentation**: Support managers couldn't see all cases across systems without manual aggregation
2. **SLA Tracking**: No automated way to measure compliance with response time SLAs (e.g., 2-hour response for high-priority cases)
3. **Priority Assignment**: Cases needed intelligent routing based on customer tier, issue severity, and current workload
4. **Data Quality**: Missing customer references, inconsistent timestamps, and duplicate entries across systems

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import List, Dict, Tuple
import logging

# Configure logging for data pipeline monitoring
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class SupportCaseAggregator:
    """
    ETL Pipeline for aggregating customer support cases from multiple dealer systems.
    Handles schema normalization, data quality validation, and SLA compliance metrics.
    """
    
    def __init__(self, sla_config: Dict[str, int]):
        """
        Args:
            sla_config: SLA response times in hours by priority level
                       e.g., {'high': 2, 'medium': 8, 'low': 24}
        """
        self.sla_config = sla_config
        self.case_df = None
        self.quality_report = {}
    
    def extract_from_sources(self, crm_data: pd.DataFrame, 
                             parts_system: pd.DataFrame,
                             warranty_system: pd.DataFrame) -> pd.DataFrame:
        """
        Extract and normalize cases from heterogeneous sources into unified schema.
        Handles schema mapping and timestamp standardization.
        """
        normalized_cases = []
        
        # CRM system extraction (Service & Key cases)
        for _, row in crm_data.iterrows():
            normalized_cases.append({
                'case_id': f"CRM-{row.get('id', 'UNKNOWN')}",
                'source': 'CRM',
                'customer_id': row.get('customer_id'),
                'dealer_location': row.get('location'),
                'case_type': row.get('case_type', 'SERVICE'),  # SERVICE, KEY_REQUEST, MAINTENANCE
                'priority': self._map_priority(row.get('severity', 'low')),
                'description': row.get('description', ''),
                'created_at': pd.to_datetime(row.get('created_date')),
                'first_response_at': pd.to_datetime(row.get('first_response')) if pd.notna(row.get('first_response')) else None,
                'resolved_at': pd.to_datetime(row.get('resolved_date')) if pd.notna(row.get('resolved_date')) else None,
                'assigned_to': row.get('assigned_agent'),
                'status': row.get('status', 'OPEN')
            })
        
        # Parts ordering system extraction
        for _, row in parts_system.iterrows():
            normalized_cases.append({
                'case_id': f"PARTS-{row.get('order_id', 'UNKNOWN')}",
                'source': 'PARTS',
                'customer_id': row.get('customer_ref'),
                'dealer_location': row.get('dealer_code'),
                'case_type': 'PARTS_ORDER',
                'priority': 'medium' if row.get('is_urgent') else 'low',
                'description': f"Part: {row.get('part_number')} - {row.get('part_description')}",
                'created_at': pd.to_datetime(row.get('order_date')),
                'first_response_at': pd.to_datetime(row.get('acknowledged_at')) if pd.notna(row.get('acknowledged_at')) else None,
                'resolved_at': pd.to_datetime(row.get('shipped_date')) if pd.notna(row.get('shipped_date')) else None,
                'assigned_to': row.get('warehouse_manager'),
                'status': row.get('fulfillment_status', 'PENDING')
            })
        
        # Warranty system extraction
        for _, row in warranty_system.iterrows():
            normalized_cases.append({
                'case_id': f"WARRANTY-{row.get('claim_id', 'UNKNOWN')}",
                'source': 'WARRANTY',
                'customer_id': row.get('customer_account'),
                'dealer_location': row.get('processing_location'),
                'case_type': 'WARRANTY_CLAIM',
                'priority': 'high' if row.get('claim_value', 0) > 1000 else 'medium',
                'description': row.get('claim_reason', ''),
                'created_at': pd.to_datetime(row.get('claim_date')),
                'first_response_at': pd.to_datetime(row.get('review_date')) if pd.notna(row.get('review_date')) else None,
                'resolved_at': pd.to_datetime(row.get('decision_date')) if pd.notna(row.get('decision_date')) else None,
                'assigned_to': row.get('claim_adjuster'),
                'status': row.get('claim_status', 'SUBMITTED')
            })
        
        self.case_df = pd.DataFrame(normalized_cases)
        logger.info(f"Extracted {len(self.case_df)} cases from {len(crm_data) + len(parts_system) + len(warranty_system)} source records")
        return self.case_df
    
    def validate_and_clean(self) -> Dict[str, int]:
        """
        Data quality validation and cleaning.
        Returns quality metrics for monitoring.
        """
        initial_count = len(self.case_df)
        
        # Detect and flag missing critical fields
        missing_customer_id = self.case_df['customer_id'].isna().sum()
        missing_created_at = self.case_df['created_at'].isna().sum()
        
        # Remove records with critical missing data
        self.case_df = self.case_df[
            (self.case_df['customer_id'].notna()) & 
            (self.case_df['created_at'].notna())
        ]
        logger.warning(f"Removed {initial_count - len(self.case_df)} records with missing critical fields")
        
        # Detect duplicates within same source (by customer_id, case_type, created_at within 1 hour)
        duplicate_mask = (
            self.case_df
            .groupby(['customer_id', 'case_type', pd.Grouper(key='created_at', freq='1H')])
            .cumcount() > 0
        )
        duplicates_found = duplicate_mask.sum()
        self.case_df = self.case_df[~duplicate_mask]
        logger.warning(f"Removed {duplicates_found} potential duplicate records")
        
        # Validate timestamp logic (resolved_at >= first_response_at >= created_at)
        invalid_timestamps = (
            (self.case_df['first_response_at'].notna()) & 
            (self.case_df['first_response_at'] < self.case_df['created_at'])
        ) | (
            (self.case_df['resolved_at'].notna()) & 
            (self.case_df['resolved_at'] < self.case_df['created_at'])
        )
        self.case_df = self.case_df[~invalid_timestamps]
        
        self.quality_report = {
            'initial_records': initial_count,
            'final_records': len(self.case_df),
            'removed_missing_fields': missing_customer_id + missing_created_at,
            'removed_duplicates': duplicates_found,
            'data_quality_score': (len(self.case_df) / initial_count * 100) if initial_count > 0 else 0
        }
        
        return self.quality_report
    
    def calculate_sla_metrics(self) -> pd.DataFrame:
        """
        Calculate SLA compliance metrics for each case.
        Identifies cases at risk of SLA breach and closed cases with SLA performance.
        """
        self.case_df['sla_hours'] = self.case_df['priority'].map(self.sla_config)
        self.case_df['sla_deadline'] = self.case_df['created_at'] + self.case_df['sla_hours'].apply(lambda x: timedelta(hours=x))
        
        # For open cases: measure time to first response
        open_mask = self.case_df['status'].isin(['OPEN', 'PENDING', 'IN_PROGRESS'])
        self.case_df.loc[open_mask, 'hours_since_creation'] = (
            (datetime.now() - self.case_df.loc[open_mask, 'created_at']).dt.total_seconds() / 3600
        )
        self.case_df.loc[open_mask, 'sla_breach_at_risk'] = (
            self.case_df.loc[open_mask, 'hours_since_creation'] > self.case_df.loc[open_mask, 'sla_hours']
        )
        
        # For closed cases: measure actual response time
        closed_mask = ~open_mask & self.case_df['first_response_at'].notna()
        self.case_df.loc[closed_mask, 'actual_response_hours'] = (
            (self.case_df.loc[closed_mask, 'first_response_at'] - 
             self.case_df.loc[closed_mask, 'created_at']).dt.total_seconds() / 3600
        )
        self.case_df.loc[closed_mask, 'sla_met'] = (
            self.case_df.loc[closed_mask, 'actual_response_hours'] <= self.case_df.loc[closed_mask, 'sla_hours']
        )
        
        return self.case_df[['case_id', 'priority', 'sla_hours', 'sla_deadline', 
                             'sla_breach_at_risk', 'sla_met', 'actual_response_hours']]
    
    def identify_escalations(self) -> pd.DataFrame:
        """
        Identify cases requiring escalation based on business rules.
        Rules: High-priority + SLA at risk, OR unresponsive cases (>5 days old with no response)
        """
        escalation_cases = []
        
        for _, case in self.case_df.iterrows():
            reasons = []
            
            # Rule 1: High priority + SLA at risk
            if (case['priority'] == 'high' and 
                hasattr(case, 'sla_breach_at_risk') and case['sla_breach_at_risk']):
                reasons.append('HIGH_PRIORITY_SLA_AT_RISK')
            
            # Rule 2: Unresponsive cases (5+ days, no first response)
            days_without_response = (datetime.now() - case['created_at']).days
            if (case['first_response_at'] is pd.NaT or pd.isna(case['first_response_at'])) and days_without_response > 5:
                reasons.append('UNRESPONSIVE_5_DAYS')
            
            # Rule 3: Long-running cases (>30 days open)
            if case['status'] in ['OPEN', 'IN_PROGRESS'] and days_without_response > 30:
                reasons.append('LONG_RUNNING_30_DAYS')
            
            if reasons:
                escalation_cases.append({
                    'case_id': case['case_id'],
                    'customer_id': case['customer_id'],
                    'case_type': case['case_type'],
                    'priority': case['priority'],
                    'created_at': case['created_at'],
                    'assigned_to': case['assigned_to'],
                    'escalation_reasons': ', '.join(reasons),
                    'days_open': days_without_response
                })
        
        escalation_df = pd.DataFrame(escalation_cases)
        logger.info(f"Identified {len(escalation_df)} cases for escalation")
        return escalation_df
    
    def _map_priority(self, severity: str) -> str:
        """Map source severity levels to standardized priority levels."""
        priority_map = {
            'critical': 'high',
            'high': 'high',
            'medium': 'medium',
            'normal': 'medium',
            'low': 'low',
            'minor': 'low'
        }
        return priority_map.get(str(severity).lower(), 'medium')
    
    def get_summary_report(self) -> Dict:
        """Generate executive summary of support metrics."""
        if self.case_df is None or len(self.case_df) == 0:
            return {}
        
        return {
            'total_open_cases': len(self.case_df[self.case_df['status'].isin(['OPEN', 'PENDING', 'IN_PROGRESS'])]),
            'total_closed_cases': len(self.case_df[~self.case_df['status'].isin(['OPEN', 'PENDING', 'IN_PROGRESS'])]),
            'sla_compliance_rate': (self.case_df['sla_met'].sum() / len(self.case_df[self.case_df['sla_met'].notna()]) * 100) if self.case_df['sla_met'].notna().any() else 0,
            'cases_at_risk': self.case_df['sla_breach_at_risk'].sum() if 'sla_breach_at_risk' in self.case_df.columns else 0,
            'avg_response_time_hours': self.case_df['actual_response_hours'].mean() if 'actual_response_hours' in self.case_df.columns else 0,
            'cases_by_priority': self.case_df['priority'].value_counts().to_dict(),
            'cases_by_source': self.case_df['source'].value_counts().to_dict()
        }

## Usage Example & Implementation

```python
# Initialize the aggregator with SLA configuration (hours for response)
sla_config = {
    'high': 2,      # High priority: 2-hour response SLA
    'medium': 8,    # Medium priority: 8-hour response SLA  
    'low': 24       # Low priority: 24-hour response SLA
}

aggregator = SupportCaseAggregator(sla_config=sla_config)

# Extract and normalize cases from three sources
unified_cases = aggregator.extract_from_sources(
    crm_data=crm_df,           # Service calls, key requests
    parts_system=parts_df,     # Parts orders with ship status
    warranty_system=warranty_df  # Warranty claims
)

# Data quality validation
quality_metrics = aggregator.validate_and_clean()
# Output: {
#   'initial_records': 15420,
#   'final_records': 15387,
#   'removed_missing_fields': 23,
#   'removed_duplicates': 10,
#   'data_quality_score': 99.79
# }

# Calculate SLA performance metrics
sla_metrics = aggregator.calculate_sla_metrics()

# Identify cases needing escalation to management
escalations = aggregator.identify_escalations()

# Generate executive summary report
summary = aggregator.get_summary_report()
# Output: {
#   'total_open_cases': 342,
#   'total_closed_cases': 8901,
#   'sla_compliance_rate': 94.3,
#   'cases_at_risk': 12,
#   'avg_response_time_hours': 3.2,
#   'cases_by_priority': {'high': 89, 'medium': 1203, 'low': 6951},
#   'cases_by_source': {'CRM': 5000, 'PARTS': 2800, 'WARRANTY': 1587}
# }
```

## Key Engineering Patterns Demonstrated

1. **Schema Normalization**: Maps heterogeneous source schemas into unified data model, enabling consistent analysis
2. **Data Quality Framework**: Validates critical fields, detects duplicates, and enforces business logic constraints
3. **Time-series SLA Calculation**: Handles both open case monitoring (at-risk detection) and closed case analysis
4. **Escalation Rules Engine**: Business rule-based identification of cases requiring management attention
5. **Logging & Observability**: Tracks data lineage and quality metrics for pipeline monitoring

## Why This Approach is Important

- **Scalability**: Pattern extends to additional source systems without rewriting core logic
- **Maintainability**: Clear separation between extraction, validation, and metrics layers
- **Auditability**: Comprehensive logging enables troubleshooting and compliance reporting  
- **Business Intelligence**: Provides data foundation for executive dashboards and SLA reporting
- **Data Governance**: Quality metrics demonstrate data fitness for decision-making